<a href="https://colab.research.google.com/github/gaurinandwana/JPMORGAN_research/blob/main/gas_storage_pricing_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
from typing import List

# ==========================================
# 1. THE PRICING MODEL FUNCTION
# ==========================================
def price_storage_contract(
    injection_dates: List[str],
    withdrawal_dates: List[str],
    injection_prices: List[float],
    withdrawal_prices: List[float],
    injection_rate: float,
    withdrawal_rate: float,
    max_storage_volume: float,
    storage_fee_per_month: float,
    injection_cost_per_unit: float,
    withdrawal_cost_per_unit: float
) -> float:
    """
    Prices a natural gas storage contract based on injection/withdrawal schedules and costs.
    Assumptions: Zero interest rates, no transport delay, continuous calendar days.
    """
    # Calculate Volumetric Cash Flows
    total_injection_cost = 0.0
    total_injected_volume = 0.0

    # Process injections
    for price in injection_prices:
        volume = injection_rate
        total_injected_volume += volume
        total_injection_cost += volume * (price + injection_cost_per_unit)

    total_withdrawal_revenue = 0.0
    total_withdrawn_volume = 0.0

    # Process withdrawals
    for price in withdrawal_prices:
        volume = withdrawal_rate
        total_withdrawn_volume += volume
        total_withdrawal_revenue += volume * (price - withdrawal_cost_per_unit)

    # Risk/Validation Guardrails
    if total_injected_volume > max_storage_volume:
        raise ValueError(f"Execution failed: Total injected volume ({total_injected_volume} MMBtu) exceeds max storage capacity ({max_storage_volume} MMBtu).")

    if total_injected_volume != total_withdrawn_volume:
        print(f"[Warning] Volume mismatch: Injected {total_injected_volume} MMBtu but withdrew {total_withdrawn_volume} MMBtu.")

    # Calculate Storage Duration & Fixed Rental Fees
    all_dates = [datetime.strptime(d, '%Y-%m-%d') for d in injection_dates + withdrawal_dates]
    start_date = min(all_dates)
    end_date = max(all_dates)

    # Estimate total months using standard 30.5 day average proxy
    total_days = (end_date - start_date).days
    total_months = max(total_days / 30.5, 1.0)

    total_storage_cost = total_months * storage_fee_per_month

    # Final Intrinsic Valuation
    contract_value = total_withdrawal_revenue - total_injection_cost - total_storage_cost

    # Breakdown report for the desk
    print("--- Valuation Breakdown ---")
    print(f"Total Volume Handled:      {total_injected_volume:,.0f} MMBtu")
    print(f"Gross Buy Cost (w/ fees):  ${total_injection_cost:,.2f}")
    print(f"Gross Sell Revenue (w/ fees): ${total_withdrawal_revenue:,.2f}")
    print(f"Estimated Storage Rent ({total_months:.2f} months): ${total_storage_cost:,.2f}")
    print("----------------------------")

    return contract_value


# ==========================================
# 2. TEST CASE CONFIGURATION (Sample Data)
# ==========================================
# The client wants to buy gas cheap in July and inject it, then withdraw/sell during winter peaks in December
test_injection_dates = ["2026-07-01", "2026-07-15"]
test_withdrawal_dates = ["2026-12-01", "2026-12-15"]

# Summer market price is ~$2.00/MMBtu; Winter spikes to ~$3.50/MMBtu
test_injection_prices = [2.00, 2.10]
test_withdrawal_prices = [3.50, 3.65]

# Operational Constraints
test_injection_rate = 500000          # 500k MMBtu injected per date entry (1M total)
test_withdrawal_rate = 500000         # 500k MMBtu withdrawn per date entry (1M total)
test_max_storage_volume = 1500000     # 1.5M MMBtu facility cap

# Pricing Friction/Fees
test_storage_fee_per_month = 100000   # $100k flat monthly rent fee
test_injection_cost_per_unit = 0.01   # $0.01 per MMBtu physical injection fee
test_withdrawal_cost_per_unit = 0.01  # $0.01 per MMBtu physical withdrawal fee


# ==========================================
# 3. RUN MODEL EXECUTION
# ==========================================
print("Running Prototype Storage Pricing Model...\n")

final_value = price_storage_contract(
    injection_dates=test_injection_dates,
    withdrawal_dates=test_withdrawal_dates,
    injection_prices=test_injection_prices,
    withdrawal_prices=test_withdrawal_prices,
    injection_rate=test_injection_rate,
    withdrawal_rate=test_withdrawal_rate,
    max_storage_volume=test_max_storage_volume,
    storage_fee_per_month=test_storage_fee_per_month,
    injection_cost_per_unit=test_injection_cost_per_unit,
    withdrawal_cost_per_unit=test_withdrawal_cost_per_unit
)

print(f"FINAL CONTRACT VALUE: ${final_value:,.2f}\n")

Running Prototype Storage Pricing Model...

--- Valuation Breakdown ---
Total Volume Handled:      1,000,000 MMBtu
Gross Buy Cost (w/ fees):  $2,060,000.00
Gross Sell Revenue (w/ fees): $3,565,000.00
Estimated Storage Rent (5.48 months): $547,540.98
----------------------------
FINAL CONTRACT VALUE: $957,459.02

